# Imbalanced learning with housing data

In this notebook, I compare multiple approaches to supervised learning on imbalanced data. I trained five models on this dataset:
1. SVM with balanced data
2. SVM with imbalanced data
3. SVM with imbalanced data using random oversampling
4. AdaBoost with imbalanced data
5. AutoML with imbalanced data

Dataset: https://www.kaggle.com/datasets/dhirajnirne/california-housing-data

In [2]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score
from sklearn.svm import SVC
from sklearn.ensemble import AdaBoostClassifier
from sklearn.model_selection import RandomizedSearchCV
from imblearn.datasets import make_imbalance
from imblearn.pipeline import Pipeline
from imblearn.over_sampling import RandomOverSampler
from scipy.stats import randint
from scipy.stats import loguniform
from tabulate import tabulate
import pickle
from tpot import TPOTClassifier

### Load the data

In [11]:
dataset = pd.read_csv("housing.csv")
dataset.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 20640 entries, 0 to 20639
Data columns (total 10 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   longitude           20640 non-null  float64
 1   latitude            20640 non-null  float64
 2   housing_median_age  20640 non-null  int64  
 3   total_rooms         20640 non-null  int64  
 4   total_bedrooms      20433 non-null  float64
 5   population          20640 non-null  int64  
 6   households          20640 non-null  int64  
 7   median_income       20640 non-null  float64
 8   median_house_value  20640 non-null  int64  
 9   ocean_proximity     20640 non-null  int64  
dtypes: float64(4), int64(6)
memory usage: 1.6 MB


In [12]:
dataset.head()

,longitude,latitude,housing_median_age,total_rooms,total_bedrooms,population,households,median_income,median_house_value,ocean_proximity
0,-122.23,37.88,41,880,129.0,322,126,8.3252,452600,3
1,-122.22,37.86,21,7099,1106.0,2401,1138,8.3014,358500,3
2,-122.24,37.85,52,1467,190.0,496,177,7.2574,352100,3
3,-122.25,37.85,52,1274,235.0,558,219,5.6431,341300,3
4,-122.25,37.85,52,1627,280.0,565,259,3.8462,342200,3


Split the data into features and labels. Use ocean_proximity as the classification label and all other columns as features.

Labels:

0. <1H OCEAN
1. INLAND
2. NEAR OCEAN
3. NEAR BAY

In [13]:
X = dataset.drop("ocean_proximity", axis=1)
y = dataset["ocean_proximity"].copy()

### Make the data imbalanced
Analyze the original data balance

In [14]:
class_frequency = np.bincount(y)
for i, frequency in enumerate(class_frequency):
    print(f"{i}: {frequency}")

0: 9136
1: 6551
2: 2663
3: 2290


Make the data more imbalanced

In [17]:
sampling_strategy = {0:9136, 1:4000, 2:1000, 3:200}
X_imb, y_imb = make_imbalance(X, y, sampling_strategy=sampling_strategy, random_state=42)
imbalanced_class_frequency = np.bincount(y_imb)
for i, frequency in enumerate(imbalanced_class_frequency):
    print(f"{i}: {frequency}")

0: 9136
1: 4000
2: 1000
3: 200


### Split the data

In [71]:
X_train, X_test, y_train, y_test = train_test_split(X, y, random_state=42)
X_train_imb, X_test_imb, y_train_imb, y_test_imb = train_test_split(X_imb, y_imb, random_state=42)

### Use TPOT to search for a model

TPOT is an AutoML library that trains a variety of random models on the dataset and chooses the best one.

In [ ]:
tpot = TPOTClassifier(max_time_mins=30)
tpot.fit(X_train_imb, y_train_imb)
pipe_tpot = tpot.fitted_pipeline_
with open("data/tpot_model.pkl", "wb") as file:
    pickle.dump(pipe_tpot, file)

c:\Users\lukek\anaconda3\envs\machine_learning\Lib\site-packages\distributed\node.py:188: UserWarning: Port 8787 is already in use.
Perhaps you already have a cluster running?
Hosting the HTTP server on port 61701 instead
  warnings.warn(
Generation: : 4it [30:04, 451.12s/it]


In [3]:
with open("data/tpot_model.pkl", "rb") as file:
    pipe_tpot = pickle.load(file)

In [4]:
pipe_tpot

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators ` for more details.","[('minmaxscaler', ...), ('variancethreshold', ...), ...]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing `.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
,"feature_range feature_range: tuple (min, max), default=(0, 1)Desired range of transformed data.","(0, ...)"
,"copy copy: bool, default=TrueSet to False to perform inplace row normalization and avoid acopy (if the input is already a numpy array).",True
,"clip clip: bool, default=FalseSet to True to clip transformed values of held-out data toprovided `feature_range`.Since this parameter will clip values, `inverse_transform` may notbe able to restore the original data... note:: Setting `clip=True` does not prevent feature drift (a distribution shift between training and test data). The transformed values are clipped to the `feature_range`, which helps avoid unintended behavior in models sensitive to out-of-range inputs (e.g. linear models). Use with care, as clipping can distort the distribution of test data... versionadded:: 0.24",False
,"threshold threshold: float, default=0Features with a training-set variance lower than this threshold willbe removed. The default is to keep all features with non-zero variance,i.e. remove the features that have the same value in all samples.",0.0019041709608
,"transformer_list transformer_list: list of (str, transformer) tuplesList of transformer objects to be applied to the data. The firsthalf of each tuple is the name of the transformer. The transformer canbe 'drop' for it to be ignored or can be 'passthrough' for features tobe passed unchanged... versionadded:: 1.1 Added the option `""passthrough""`... versionchanged:: 0.22 Deprecated `None` as a transformer in favor of 'drop'.","[('skiptransformer', ...), ('passthrough', ...)]"
,"n_jobs n_jobs: int, default=NoneNumber of jobs to run in parallel.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary `for more details... versionchanged:: v0.20 `n_jobs` default changed from 1 to None",None
,"transformer_weights transformer_weights: dict, default=NoneMultiplicative weights for features per transformer.Keys are transformer names, values the weights.Raises ValueError if key not present in ``transformer_list``.",None


### Define the pipelines

Organize models with the pipeline from the imbalanced-learn library (scikit-learn also offers a pipeline that works the same way, but it's not compatible with oversampling).

In [75]:
pipe_svm = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
    ("classifier", SVC())
])

pipe_oversampling = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("oversampler", RandomOverSampler(random_state=42)),
    ("scaler", StandardScaler()),
    ("classifier", SVC())
])

pipe_adaboost = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
    ("classifier", AdaBoostClassifier())
])

### Define hyperparameters to test during random search

In [76]:
hyperparameters_svm = {
    "classifier__C": loguniform(0.1, 10)
}

hyperparameters_adaboost = {
    "classifier__n_estimators": randint(20, 80),
    "classifier__learning_rate": loguniform(0.1, 10)
}

### Training and Testing
Rather than training and testing the models one by one, I wrote a class to store each model along with a method to perform all the steps needed to train and evaluate them. Later on, this allows me to train and test all the models at once in a single for loop.

In [ ]:
class Model:
    def __init__(self, name, pipeline, hyperparameters, X_train, y_train, X_test, y_test):
        self.name = name
        self.pipeline = pipeline
        self.hyperparameters = hyperparameters
        self.X_train = X_train
        self.y_train = y_train
        self.X_test = X_test
        self.y_test = y_test

    def train_and_test(self):
        rs = RandomizedSearchCV(self.pipeline, self.hyperparameters, n_jobs=-1, random_state=42)
        rs.fit(self.X_train, self.y_train)
        y_pred = rs.predict(self.X_test)

        scores = [
            f1_score(self.y_test, y_pred, average="micro"),
            f1_score(self.y_test, y_pred, average="macro")
        ]

        return scores

class TPOTModel:
    def __init__(self, name, pipeline, X_test, y_test):
        self.name = name
        self.pipeline = pipeline
        self.X_test = X_test
        self.y_test = y_test

    def train_and_test(self):
        # The TPOT model is already, trained, so we only need to test it.
        y_pred = self.pipeline.predict(self.X_test)

        scores = [
            f1_score(self.y_test, y_pred, average="micro"),
            f1_score(self.y_test, y_pred, average="macro")
        ]

        return scores

Define the models and store them all in an array, indicating whether to use them on the balanced or imbalanced dataset

In [78]:
models = [
    Model("SVM (balanced)", pipe_svm, hyperparameters_svm, X_train, y_train, X_test, y_test),
    Model("SVM (imbalanced)", pipe_svm, hyperparameters_svm, X_train_imb, y_train_imb, X_test_imb, y_test_imb),
    Model("SVM with oversampling", pipe_oversampling, hyperparameters_svm, X_train_imb, y_train_imb, X_test_imb, y_test_imb),
    Model("AdaBoost", pipe_adaboost, hyperparameters_adaboost, X_train_imb, y_train_imb, X_test_imb, y_test_imb),
    TPOTModel("TPOT", pipe_tpot, X_test_imb, y_test_imb)
]

Train and test each model

In [79]:
scores = [m.train_and_test() for m in models]

c:\Users\lukek\anaconda3\envs\machine_learning\Lib\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


### Results

In [ ]:
column_titles = ["Micro F1-score", "Macro F1-score"]
row_titles = [model.name for model in models]
rounded_scores = np.around(scores, 3)
print(tabulate(rounded_scores, headers=column_titles, tablefmt="fancy_grid", showindex=row_titles))

╒═══════════════════════╤══════════════════╤══════════════════╕
│                       │   Micro F1-score │   Macro F1-score │
╞═══════════════════════╪══════════════════╪══════════════════╡
│ SVM (balanced)        │            0.89  │            0.861 │
├───────────────────────┼──────────────────┼──────────────────┤
│ SVM (imbalanced)      │            0.927 │            0.74  │
├───────────────────────┼──────────────────┼──────────────────┤
│ SVM with oversampling │            0.885 │            0.742 │
├───────────────────────┼──────────────────┼──────────────────┤
│ AdaBoost              │            0.876 │            0.737 │
├───────────────────────┼──────────────────┼──────────────────┤
│ TPOT                  │            0.982 │            0.954 │
╘═══════════════════════╧══════════════════╧══════════════════╛


### Conclusions

- The best model was the one found by TPOT. However, searching for a model with TPOT is computationally expensive.
- Curiously, the SVM didn't perform noticeably better on the imbalanced dataset after applying oversampling.